In [1]:
# Importing the Libraries
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.preprocessing import RobustScaler, LabelEncoder

# load the dataset
dataset = pd.read_excel('BBDM Project.xlsx')

# Encode the target variable
encoder = LabelEncoder()
dataset['RMC'] = encoder.fit_transform(dataset['RMC'])
original_class_names = encoder.classes_  # Store original class names

# Prepare Features and Target
X = dataset.drop(columns='RMC', axis=1)
Y = dataset['RMC']


# Train-Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Scale Features
scaler = RobustScaler()
scaler.fit(X_train)  # Fit on the training data
X_train_standardized = scaler.transform(X_train)
X_test_standardized = scaler.transform(X_test)

# Define the BaggingClassifier
model = BaggingClassifier()

# Define the grid of hyperparameters
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False],
}

# Define cross-validation strategy
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# Perform grid search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=cv, scoring='accuracy', error_score=0)
grid_result = grid_search.fit(X_train_standardized, Y_train)

# Summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.864076 using {'bootstrap': True, 'bootstrap_features': True, 'max_features': 1.0, 'max_samples': 0.7, 'n_estimators': 100}
0.816770 (0.016455) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.5, 'n_estimators': 10}
0.835848 (0.012702) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.5, 'n_estimators': 50}
0.837361 (0.011370) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.5, 'n_estimators': 100}
0.818524 (0.016641) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.7, 'n_estimators': 10}
0.834336 (0.010385) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.7, 'n_estimators': 50}
0.837241 (0.011010) with: {'bootstrap': True, 'bootstrap_features': True, 'max_features': 0.5, 'max_samples': 0.7, 'n_estimators': 100}
0.814165 (0.018820) with: {'bootstrap': True, 'bootstrap_